<a href="https://colab.research.google.com/github/komalprashar790-cmyk/Machine-Learning-model/blob/main/randomforestregressor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from os import pread
from pandas.core.arrays import categorical
# importing libraries
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV # Import GridSearchCV
from sklearn.pipeline import Pipeline # Import Pipeline
from sklearn.impute import SimpleImputer # Import SimpleImputer

# modeling
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score

#setup random seed
np.random.seed(42)

# import data and drop rows with missing labels
data = pd.read_csv("/content/starbucks_customer_ordering_patterns.csv.xls")
print(data.head())
data.dropna(subset=["customer_id","order_id","store_id","order_date", "order_time", "day_of_week"],inplace=True)

#define different feature and transformer pipepline
categorical_features = ["order_channel","customer_gender","customer_age_group"]

categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
                          ('onehot', OneHotEncoder(handle_unknown='ignore'))])

numeric_features = ["total_spend", "cart_size"]
numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='mean'))])

#set up preprocessing steps (fill missing values, then connect to numbers)
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features),
        ('num', numeric_transformer, numeric_features)
    ])

# creating a preprocessing and modeling pipeline
model = Pipeline(steps=[('preprocessor', preprocessor),
                        ('model', RandomForestRegressor())])

# split data
x = data.drop("customer_satisfaction",axis=1)
y = data["customer_satisfaction"]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)


#fit and score the model
model.fit(x_train,y_train)
model.score(x_test,y_test)


# use gridSearchCV with our regression pipeline

pipe_grid = {"preprocessor__num__imputer__strategy":["mean","median"],
             "model__n_estimators":[100,250],
             "model__max_depth":[None,3],
             "model__max_features":["sqrt"],
             "model__min_samples_split":[2,3]
}

gs_grid = GridSearchCV(model,pipe_grid,cv=5,verbose=2)
gs_grid.fit(x_train,y_train)
gs_grid.score(x_test,y_test)

  customer_id      order_id  order_date order_time day_of_week order_channel  \
0  CUST_12974  ORD_00000001  25-03-2024      08:47         Mon    Drive-Thru   
1  CUST_08235  ORD_00000002  18-07-2025      08:02         Fri    Mobile App   
2  CUST_00393  ORD_00000003  15-01-2025      05:40         Wed         Kiosk   
3  CUST_06936  ORD_00000004  30-07-2024      15:10         Tue    Drive-Thru   
4  CUST_09800  ORD_00000005  18-06-2024      07:38         Tue    Drive-Thru   

  store_id store_location_type     region customer_age_group customer_gender  \
0  STR_340            Suburban  Southwest              18-24            Male   
1  STR_425               Urban  Northeast              35-44          Female   
2  STR_103            Suburban    Midwest              25-34          Female   
3  STR_318            Suburban    Midwest              25-34          Female   
4  STR_338            Suburban  Northeast              35-44          Female   

   is_rewards_member  cart_size  num_c

0.01941283575466024

# New Section